# Muria: AfriqueGemma-4B Adapter Merge & GGUF Quantization

This dedicated export notebook pulls the trained LoRA adapter from Hugging Face (`fallback-ai/Muria-Afrique-Gemma-4B-LoRA`), merges it with the base model `McGill-NLP/AfriqueGemma-4B`, converts the merged weights to **GGUF (Q4_K_M)** via `llama.cpp`, verifies output with a test prompt, and publishes the GGUF models directly to Hugging Face (`fallback-ai/Muria-Afrique-Gemma-4B`).

## Environment Setup & Dependency Installation

In [ ]:

!pip install -q -U torchao transformers accelerate peft huggingface_hub gguf sentencepiece


## Authentication & Configuration

In [ ]:
import os
import gc
import torch
from huggingface_hub import HfApi, login
from kaggle_secrets import UserSecretsClient

print("Authenticating with Hugging Face...")
try:
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN2")
except Exception:
    hf_token = os.environ.get("HF_TOKEN2") or os.environ.get("HF_TOKEN")

login(token=hf_token)
api = HfApi(token=hf_token)

# Repository and Directory Configuration
BASE_MODEL = "McGill-NLP/AfriqueGemma-4B"
LORA_REPO = "fallback-ai/Muria-Afrique-Gemma-4B-LoRA"
TARGET_REPO = "fallback-ai/Muria-Afrique-Gemma-4B"

WORKING_DIR = "/kaggle/working"
MERGED_DIR = f"{WORKING_DIR}/muria-afriquegemma-4b-faw-merged"
GGUF_DIR = f"{WORKING_DIR}/muria_gguf_export"

for d in [MERGED_DIR, GGUF_DIR]:
    os.makedirs(d, exist_ok=True)

print(f"Base Model:  {BASE_MODEL}")
print(f"LoRA Source: {LORA_REPO}")
print(f"Target Repo: {TARGET_REPO}")


## Merge LoRA Adapters into Base Model (16-Bit)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

print(f"Loading base tokenizer: {BASE_MODEL}...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)

print(f"Loading base model: {BASE_MODEL} on CPU for safe weight merging...")
# Loading on CPU avoids GPU VRAM allocation pressure when loading full float16 weights
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="cpu",
    trust_remote_code=True
)

print(f"Loading LoRA adapter from {LORA_REPO}...")
model = PeftModel.from_pretrained(base_model, LORA_REPO)

print("Merging LoRA adapter weights into base model...")
merged_model = model.merge_and_unload()

print(f"Saving merged standalone model to: {MERGED_DIR}...")
merged_model.save_pretrained(MERGED_DIR, safe_serialization=True)
tokenizer.save_pretrained(MERGED_DIR)

# Free memory
del base_model
del model
del merged_model
gc.collect()
print("Merged model successfully saved.")


## Setup llama.cpp & Convert to F16 GGUF

In [ ]:
# Clone llama.cpp repository and fetch prebuilt release binaries
!git clone --depth 1 https://github.com/ggerganov/llama.cpp /kaggle/working/llama.cpp
!wget -q "https://github.com/ggml-org/llama.cpp/releases/download/b10107/llama-b10107-bin-ubuntu-x64.tar.gz" -O /kaggle/working/llama-release.tar.gz
!mkdir -p /kaggle/working/llama-release
!tar -xzf /kaggle/working/llama-release.tar.gz -C /kaggle/working/llama-release

LLAMA_BIN_DIR = "/kaggle/working/llama-release/llama-b10107"
!chmod +x {LLAMA_BIN_DIR}/llama-quantize {LLAMA_BIN_DIR}/llama-cli

F16_GGUF = f"{GGUF_DIR}/muria-afriquegemma-4b-faw-f16.gguf"
print(f"Converting {MERGED_DIR} to F16 GGUF -> {F16_GGUF}...")


!python /kaggle/working/llama.cpp/convert_hf_to_gguf.py {MERGED_DIR} \
    --outfile {F16_GGUF} \
    --outtype f16


## Quantize to Q4_K_M for Android Mobile Edge Deployment

In [ ]:
Q4_GGUF = f"{GGUF_DIR}/muria-afriquegemma-4b-faw-q4_k_m.gguf"
print(f"Quantizing F16 GGUF to Q4_K_M -> {Q4_GGUF}...")

!{LLAMA_BIN_DIR}/llama-quantize {F16_GGUF} {Q4_GGUF} Q4_K_M
!ls -lh {F16_GGUF} {Q4_GGUF}


## Quick Verification Inference Test

In [ ]:
# Run sample test using Gemma turn format
test_prompt = (
    "<start_of_turn>user\n"
    "What pesticide dosage should I mix in my knapsack sprayer for Fall Armyworm?<end_of_turn>\n"
    "<start_of_turn>model\n"
)

!{LLAMA_BIN_DIR}/llama-cli -m {Q4_GGUF} \
    -p "{test_prompt}" \
    -n 150


## Publish GGUF Quantized Models & Model Card to Hugging Face

In [ ]:
api.create_repo(repo_id=TARGET_REPO, repo_type="model", exist_ok=True, private=True)

# Upload Q4_K_M GGUF (primary mobile inference asset-option)
if os.path.exists(Q4_GGUF):
    size_gb = os.path.getsize(Q4_GGUF) / (1024 ** 3)
    print(f"Uploading {Q4_GGUF} ({size_gb:.2f} GB) to {TARGET_REPO}...")
    api.upload_file(
        path_or_fileobj=Q4_GGUF,
        path_in_repo="muria-afriquegemma-4b-faw-q4_k_m.gguf",
        repo_id=TARGET_REPO,
        repo_type="model"
    )
    print("Q4_K_M GGUF uploaded successfully.")

# Upload F16 GGUF
if os.path.exists(F16_GGUF):
    size_gb = os.path.getsize(F16_GGUF) / (1024 ** 3)
    print(f"Uploading {F16_GGUF} ({size_gb:.2f} GB) to {TARGET_REPO}...")
    api.upload_file(
        path_or_fileobj=F16_GGUF,
        path_in_repo="muria-afriquegemma-4b-faw-f16.gguf",
        repo_id=TARGET_REPO,
        repo_type="model"
    )
    print("F16 GGUF uploaded successfully.")

# Upload MODEL_CARD.md if found
for card_path in ["/kaggle/input/datasets/jodinho/model-card/MODEL_CARD.md", "MODEL_CARD.md"]:
    if os.path.exists(card_path):
        print(f"Uploading model card from {card_path} as README.md...")
        api.upload_file(
            path_or_fileobj=card_path,
            path_in_repo="README.md",
            repo_id=TARGET_REPO,
            repo_type="model"
        )
        print("README.md uploaded successfully.")
        break

print(f"\nAll exports complete: https://huggingface.co/{TARGET_REPO}")


In [ ]:
# CONVERT TO CACTUS (.cact) BINARY & PUBLISH TO HUGGING FACE

import os
import sys
import glob
import subprocess
import shutil
from huggingface_hub import HfApi, login, snapshot_download
from kaggle_secrets import UserSecretsClient

print("--- Starting Cactus (.cact) Export Pipeline ---")

# 1. Directory Configuration
WORKING_DIR = "/kaggle/working"
CACTUS_SRC_DIR = f"{WORKING_DIR}/cactus_src"
CACT_EXPORT_DIR = f"{WORKING_DIR}/muria_cact_export"
MERGED_DIR = f"{WORKING_DIR}/muria-afriquegemma-4b-faw-merged"
BASE_MODEL = "McGill-NLP/AfriqueGemma-4B"
LORA_ADAPTER = "fallback-ai/Muria-Afrique-Gemma-4B-LoRA"
TARGET_HF_REPO = "fallback-ai/Muria-Afrique-Gemma-4B"

os.makedirs(CACT_EXPORT_DIR, exist_ok=True)

# Remove any old shadowing /kaggle/working/cactus directory from previous failed runs
old_shadowing_dir = f"{WORKING_DIR}/cactus"
if os.path.exists(old_shadowing_dir):
    print(f"Removing shadowing directory: {old_shadowing_dir}...")
    shutil.rmtree(old_shadowing_dir, ignore_errors=True)

# 2. FIX: Upgrade torchao and peft BEFORE any PEFT/LoRA operations.
# Kaggle pre-installs torchao 0.10.0. PEFT's LoRA dispatcher now requires >= 0.16.0.
# The fix MUST happen in this subprocess so importlib cache is refreshed on next use.
print("Upgrading torchao and peft to compatible versions...")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-U", "torchao", "peft"],
    check=True
)
print("torchao + peft upgraded successfully.")

# 3. Authentication with Hugging Face Hub
print("Authenticating with Hugging Face Hub...")
try:
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN2")
except Exception:
    hf_token = os.environ.get("HF_TOKEN2") or os.environ.get("HF_TOKEN")

assert hf_token, "CRITICAL: HF_TOKEN2 not found in Kaggle Secrets or environment!"
login(token=hf_token, add_to_git_credential=False)
api = HfApi(token=hf_token)

# 4. Clone Cactus to isolated cactus_src & Install Python Package
if not os.path.exists(CACTUS_SRC_DIR):
    print("Cloning Cactus repository into cactus_src...")
    subprocess.run(
        ["git", "clone", "--depth", "1", "https://github.com/cactus-compute/cactus", CACTUS_SRC_DIR],
        check=True
    )
else:
    print("Cactus repository already present in cactus_src.")

print("Installing Cactus Python package from cactus_src/python...")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", f"{CACTUS_SRC_DIR}/python"],
    check=True
)
print("Cactus CLI installed successfully.")

# 5. Prepare Model Weights for Conversion
use_local_merged = os.path.exists(MERGED_DIR) and len(os.listdir(MERGED_DIR)) > 0

cli_bin = shutil.which("cactus") or "cactus"

if use_local_merged:
    print(f"Using pre-merged 16-bit standalone model from: {MERGED_DIR}")
    model_source = MERGED_DIR
    lora_arg = []
else:
    print(f"Local merged directory not found. Using Base ({BASE_MODEL}) + LoRA ({LORA_ADAPTER})...")
    # Download private LoRA adapter locally first — Cactus CLI cannot auth private Hub repos for --lora.
    local_lora_dir = f"{WORKING_DIR}/muria_lora_local"
    if not os.path.exists(local_lora_dir) or len(os.listdir(local_lora_dir)) == 0:
        print(f"Downloading authenticated LoRA adapter to {local_lora_dir}...")
        snapshot_download(
            repo_id=LORA_ADAPTER,
            local_dir=local_lora_dir,
            token=hf_token
        )
        print("LoRA adapter downloaded successfully.")
    else:
        print(f"LoRA adapter already present at {local_lora_dir}.")
    model_source = BASE_MODEL
    lora_arg = ["--lora", local_lora_dir]

# 6. Build and run the conversion command
convert_cmd = [
    cli_bin, "convert",
    model_source,
    CACT_EXPORT_DIR,
    "--bits", "4",
    "--token", hf_token
] + lora_arg

safe_cmd = [c if not str(c).startswith("hf_") else "hf_***" for c in convert_cmd]
print("Executing Cactus conversion...")
print("Command:", " ".join(safe_cmd))

env = os.environ.copy()
env["PYTHONPATH"] = f"{CACTUS_SRC_DIR}/python:" + env.get("PYTHONPATH", "")
env["HF_TOKEN"] = hf_token
env["HUGGING_FACE_HUB_TOKEN"] = hf_token

res = subprocess.run(convert_cmd, env=env)
if res.returncode != 0:
    print(f"Primary CLI call exited with code {res.returncode}. Retrying with python -m cactus...")
    fallback_cmd = [
        sys.executable, "-m", "cactus", "convert",
        model_source,
        CACT_EXPORT_DIR,
        "--bits", "4",
        "--token", hf_token
    ] + lora_arg
    subprocess.run(fallback_cmd, check=True, env=env)

print("Cactus conversion completed successfully!")

# 7. Locate Generated .cact Files
search_paths = [
    f"{CACT_EXPORT_DIR}/**/*.cact",
    f"{CACT_EXPORT_DIR}/*.cact",
    f"{WORKING_DIR}/**/*.cact",
    f"{CACTUS_SRC_DIR}/**/*.cact"
]

found_cact = []
for p in search_paths:
    found_cact.extend(glob.glob(p, recursive=True))

found_cact = list(set(found_cact))
assert len(found_cact) > 0, (
    f"CRITICAL ERROR: No .cact file found after conversion! "
    f"Checked directories: {CACT_EXPORT_DIR}, {WORKING_DIR}"
)

cact_file = found_cact[0]
file_size_gb = os.path.getsize(cact_file) / (1024 ** 3)
target_name = "muria-afriquegemma-4b-faw.cact"

print(f"\nFOUND: {cact_file}")
print(f"Size: {file_size_gb:.2f} GB")

# 8. Upload to Hugging Face Hub
print(f"\nUploading {target_name} to https://huggingface.co/{TARGET_HF_REPO}...")
api.create_repo(repo_id=TARGET_HF_REPO, repo_type="model", exist_ok=True, private=True)

api.upload_file(
    path_or_fileobj=cact_file,
    path_in_repo=target_name,
    repo_id=TARGET_HF_REPO,
    repo_type="model"
)

api.upload_file(
    path_or_fileobj=cact_file,
    path_in_repo=f"cact/{target_name}",
    repo_id=TARGET_HF_REPO,
    repo_type="model"
)

print(f"\nUpload complete: https://huggingface.co/{TARGET_HF_REPO}")
